In [ ]:
# Imports and Configuration
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import os

# Plot styling for publication
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['font.size'] = 12
plt.rcParams['axes.labelsize'] = 14
plt.rcParams['axes.titlesize'] = 16
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['figure.dpi'] = 150

# Color palettes
METHOD_COLORS = {'LDA': '#1f77b4', 'PCA': '#ff7f0e', 'Full': '#2ca02c', 'RP': '#d62728'}
BACKBONE_COLORS = {'ResNet-18': '#1f77b4', 'ResNet-50': '#ff7f0e', 
                   'MobileNetV3-Small': '#2ca02c', 'EfficientNet-B0': '#d62728'}

# Paths
RESULTS_DIR = '../results'
FIGURE_DIR = '../figures'
os.makedirs(FIGURE_DIR, exist_ok=True)

# Display settings
pd.set_option('display.precision', 4)
pd.set_option('display.max_columns', 20)
%matplotlib inline

print("Setup complete!")

---
## 1. Load All Results

In [ ]:
# Load all result files
print("Loading results...")
print("="*60)

# Basic CIFAR-100 results
cifar_results = pd.read_csv(f'{RESULTS_DIR}/results.csv')
print(f"✓ CIFAR-100 basic results: {len(cifar_results)} experiments")

# Backbone comparison
backbone_results = pd.read_csv(f'{RESULTS_DIR}/backbone_comparison.csv')
print(f"✓ Backbone comparison: {len(backbone_results)} experiments")

# Statistical significance
sig_results = pd.read_csv(f'{RESULTS_DIR}/statistical_significance/multi_seed_results.csv')
sig_summary = pd.read_csv(f'{RESULTS_DIR}/statistical_significance/significance_summary.csv')
print(f"✓ Statistical significance: {len(sig_results)} seed runs")

# Data efficiency
efficiency_results = pd.read_csv(f'{RESULTS_DIR}/data_efficiency/data_efficiency_results.csv')
print(f"✓ Data efficiency: {len(efficiency_results)} experiments")

# Tiny ImageNet (if available)
try:
    tiny_backbone = pd.read_csv(f'{RESULTS_DIR}/tiny_imagenet/backbone_comparison.csv')
    print(f"✓ Tiny ImageNet: {len(tiny_backbone)} experiments")
except:
    tiny_backbone = None
    print("  Tiny ImageNet results not available")

print("\nAll results loaded successfully!")

---
## 2. Statistical Significance Analysis

**Key Question**: Is the LDA improvement statistically significant, not due to random chance?

In [ ]:
# Display significance summary
print("STATISTICAL SIGNIFICANCE SUMMARY")
print("="*80)
print("\nAccuracy: mean ± std across 5 seeds")
print("-"*80)

for idx, row in sig_summary.iterrows():
    dataset = row['dataset']
    print(f"\n{dataset}:")
    print(f"  Full Features: {row['acc_full_mean']:.4f} ± {row['acc_full_std']:.4f}")
    print(f"  LDA ({int(row['n_components'])} comp): {row['acc_lda_mean']:.4f} ± {row['acc_lda_std']:.4f}")
    print(f"  PCA ({int(row['n_components'])} comp): {row['acc_pca_mean']:.4f} ± {row['acc_pca_std']:.4f}")
    print(f"\n  LDA vs Full improvement: {row['lda_vs_full_mean']*100:+.2f}%")
    print(f"  t-test p-value: {row['t_pvalue_lda_vs_full']:.6f}")
    print(f"  Wilcoxon p-value: {row['wilcoxon_pvalue_lda_vs_full']:.6f}")
    
    if row['wilcoxon_pvalue_lda_vs_full'] < 0.05:
        print(f"  → SIGNIFICANT at p<0.05 ✓")
    else:
        print(f"  → Not significant at p<0.05")

In [ ]:
# Visualization: Multi-seed comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Filter for CIFAR-100
cifar_sig = sig_results[sig_results['dataset'] == 'cifar100']

# Left: Bar chart with error bars
ax1 = axes[0]
methods = ['Full', 'LDA', 'PCA']
means = [cifar_sig['acc_full'].mean(), cifar_sig['acc_lda'].mean(), cifar_sig['acc_pca'].mean()]
stds = [cifar_sig['acc_full'].std(), cifar_sig['acc_lda'].std(), cifar_sig['acc_pca'].std()]
colors = [METHOD_COLORS['Full'], METHOD_COLORS['LDA'], METHOD_COLORS['PCA']]

bars = ax1.bar(methods, means, yerr=stds, capsize=5, color=colors, alpha=0.8, edgecolor='black')
ax1.set_ylabel('Test Accuracy')
ax1.set_title('CIFAR-100: Method Comparison (5 seeds)')
ax1.set_ylim(0.35, 0.45)

# Add value labels
for bar, mean, std in zip(bars, means, stds):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + std + 0.005, 
             f'{mean:.4f}', ha='center', va='bottom', fontsize=11)

# Right: Seed-by-seed comparison
ax2 = axes[1]
x = np.arange(len(cifar_sig))
width = 0.25

ax2.bar(x - width, cifar_sig['acc_full'], width, label='Full', color=METHOD_COLORS['Full'], alpha=0.8)
ax2.bar(x, cifar_sig['acc_lda'], width, label='LDA', color=METHOD_COLORS['LDA'], alpha=0.8)
ax2.bar(x + width, cifar_sig['acc_pca'], width, label='PCA', color=METHOD_COLORS['PCA'], alpha=0.8)

ax2.set_xlabel('Seed')
ax2.set_ylabel('Test Accuracy')
ax2.set_title('CIFAR-100: Accuracy by Seed')
ax2.set_xticks(x)
ax2.set_xticklabels(cifar_sig['seed'].values)
ax2.legend()
ax2.set_ylim(0.35, 0.45)

plt.tight_layout()
plt.savefig(f'{FIGURE_DIR}/statistical_significance.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nFigure saved: figures/statistical_significance.png")

---
## 3. Training Data Efficiency Analysis

**Key Question**: Does LDA advantage increase with limited training data?

In [ ]:
# Aggregate data efficiency results
eff_summary = efficiency_results.groupby(['dataset', 'train_fraction']).agg({
    'n_train_samples': 'first',
    'acc_full': ['mean', 'std'],
    'acc_lda': ['mean', 'std'],
    'acc_pca': ['mean', 'std'],
    'lda_vs_full': ['mean', 'std']
}).reset_index()

# Flatten column names
eff_summary.columns = ['_'.join(col).strip('_') if isinstance(col, tuple) else col 
                       for col in eff_summary.columns]

print("DATA EFFICIENCY ANALYSIS")
print("="*80)
print("\nCIFAR-100 Results:")
print("-"*80)
print(f"{'Fraction':<12} {'Samples':<10} {'Full':<18} {'LDA':<18} {'LDA Gain':<12}")
print("-"*80)

cifar_eff = eff_summary[eff_summary['dataset'] == 'cifar100']
for _, row in cifar_eff.iterrows():
    frac = f"{row['train_fraction']*100:.0f}%"
    samples = f"{int(row['n_train_samples_first']):,}" if 'n_train_samples_first' in row else f"{int(row['n_train_samples']):,}"
    full = f"{row['acc_full_mean']:.4f}±{row['acc_full_std']:.4f}"
    lda = f"{row['acc_lda_mean']:.4f}±{row['acc_lda_std']:.4f}"
    gain = f"{row['lda_vs_full_mean']*100:+.2f}%"
    print(f"{frac:<12} {samples:<10} {full:<18} {lda:<18} {gain:<12}")

In [ ]:
# Visualization: Data Efficiency
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cifar_eff = eff_summary[eff_summary['dataset'] == 'cifar100'].sort_values('train_fraction')

# Left: Accuracy vs Training Data Fraction
ax1 = axes[0]
fractions = cifar_eff['train_fraction'] * 100

ax1.errorbar(fractions, cifar_eff['acc_full_mean'], yerr=cifar_eff['acc_full_std'], 
             fmt='o-', color=METHOD_COLORS['Full'], label='Full Features', 
             linewidth=2, markersize=10, capsize=5)
ax1.errorbar(fractions, cifar_eff['acc_lda_mean'], yerr=cifar_eff['acc_lda_std'], 
             fmt='s-', color=METHOD_COLORS['LDA'], label='LDA (99 comp)', 
             linewidth=2, markersize=10, capsize=5)
ax1.errorbar(fractions, cifar_eff['acc_pca_mean'], yerr=cifar_eff['acc_pca_std'], 
             fmt='^-', color=METHOD_COLORS['PCA'], label='PCA (99 comp)', 
             linewidth=2, markersize=10, capsize=5)

ax1.set_xlabel('Training Data (%)')
ax1.set_ylabel('Test Accuracy')
ax1.set_title('CIFAR-100: Accuracy vs Training Data Amount')
ax1.legend(loc='lower right')
ax1.set_xticks([10, 25, 50, 100])
ax1.grid(True, alpha=0.3)

# Right: LDA Advantage vs Training Data
ax2 = axes[1]
gains = cifar_eff['lda_vs_full_mean'] * 100
gain_stds = cifar_eff['lda_vs_full_std'] * 100 if 'lda_vs_full_std' in cifar_eff.columns else np.zeros(len(gains))

bars = ax2.bar(fractions, gains, width=8, color=METHOD_COLORS['LDA'], alpha=0.8, edgecolor='black')
ax2.axhline(y=0, color='gray', linestyle='--', linewidth=1)

# Color bars based on positive/negative
for bar, gain in zip(bars, gains):
    if gain < 0:
        bar.set_color('#d62728')  # Red for negative

ax2.set_xlabel('Training Data (%)')
ax2.set_ylabel('LDA Gain over Full Features (% points)')
ax2.set_title('CIFAR-100: LDA Advantage by Data Amount')
ax2.set_xticks([10, 25, 50, 100])
ax2.grid(True, alpha=0.3, axis='y')

# Add value labels
for bar, gain in zip(bars, gains):
    ypos = bar.get_height() + 0.3 if gain > 0 else bar.get_height() - 0.8
    ax2.text(bar.get_x() + bar.get_width()/2, ypos, f'{gain:+.1f}%', 
             ha='center', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig(f'{FIGURE_DIR}/data_efficiency.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nFigure saved: figures/data_efficiency.png")

In [ ]:
# Key Finding: Does LDA advantage change with data amount?
print("\nKEY FINDING: LDA Advantage vs Training Data Amount")
print("="*60)

cifar_eff = eff_summary[eff_summary['dataset'] == 'cifar100'].sort_values('train_fraction')
gains = cifar_eff['lda_vs_full_mean'].values * 100
fractions = cifar_eff['train_fraction'].values * 100

print(f"\nAt 10% data:  LDA gain = {gains[0]:+.2f}%")
print(f"At 25% data:  LDA gain = {gains[1]:+.2f}%")
print(f"At 50% data:  LDA gain = {gains[2]:+.2f}%")
print(f"At 100% data: LDA gain = {gains[3]:+.2f}%")

# Analysis
if gains[0] < 0 and gains[-1] > 0:
    print("\n→ LDA UNDERPERFORMS with very limited data (10%)")
    print("→ LDA advantage INCREASES as more training data becomes available")
    print("→ This suggests LDA needs sufficient samples to estimate class covariances reliably")
elif gains[0] > gains[-1]:
    print("\n→ LDA advantage is LARGER with limited data")
    print("→ LDA acts as effective regularization in low-data regimes")
else:
    print("\n→ LDA advantage is relatively STABLE across data amounts")

---
## 4. Backbone Comparison Analysis

In [ ]:
# Analyze backbone results
print("BACKBONE COMPARISON ANALYSIS")
print("="*80)

# Get full features baseline for each backbone
full_baseline = backbone_results[backbone_results['method'] == 'None (Full)'].copy()
full_baseline = full_baseline.groupby('backbone')['accuracy'].max().to_dict()

# Get best LDA for each backbone
lda_results = backbone_results[backbone_results['method'] == 'LDA'].copy()
best_lda = lda_results.groupby('backbone')['accuracy'].max().to_dict()
best_lda_comp = lda_results.loc[lda_results.groupby('backbone')['accuracy'].idxmax()].set_index('backbone')['components'].to_dict()

# Get best PCA for each backbone
pca_results = backbone_results[backbone_results['method'] == 'PCA'].copy()
best_pca = pca_results.groupby('backbone')['accuracy'].max().to_dict()

# Display comparison
print(f"\n{'Backbone':<20} {'Feature Dim':<12} {'Full':<10} {'Best LDA':<15} {'LDA Gain':<12}")
print("-"*80)

for backbone in ['ResNet-18', 'MobileNetV3-Small', 'EfficientNet-B0', 'ResNet-50']:
    if backbone in full_baseline:
        dim = backbone_results[backbone_results['backbone'] == backbone]['feature_dim'].iloc[0]
        full_acc = full_baseline[backbone]
        lda_acc = best_lda[backbone]
        lda_comp = best_lda_comp[backbone]
        gain = (lda_acc - full_acc) * 100
        
        print(f"{backbone:<20} {dim:<12} {full_acc:.4f}    {lda_acc:.4f} ({int(lda_comp)}c)  {gain:+.2f}%")

In [ ]:
# Visualization: Backbone comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

backbones = ['ResNet-18', 'MobileNetV3-Small', 'EfficientNet-B0', 'ResNet-50']
dims = [512, 576, 1280, 2048]

# Left: Full vs LDA comparison
ax1 = axes[0]
x = np.arange(len(backbones))
width = 0.35

full_accs = [full_baseline.get(b, 0) for b in backbones]
lda_accs = [best_lda.get(b, 0) for b in backbones]

bars1 = ax1.bar(x - width/2, [a*100 for a in full_accs], width, label='Full Features', color=METHOD_COLORS['Full'], alpha=0.8)
bars2 = ax1.bar(x + width/2, [a*100 for a in lda_accs], width, label='Best LDA', color=METHOD_COLORS['LDA'], alpha=0.8)

ax1.set_xlabel('Backbone')
ax1.set_ylabel('Test Accuracy (%)')
ax1.set_title('CIFAR-100: Full Features vs LDA by Backbone')
ax1.set_xticks(x)
ax1.set_xticklabels([f"{b}\n({d}D)" for b, d in zip(backbones, dims)], fontsize=10)
ax1.legend()
ax1.set_ylim(50, 75)

# Right: LDA improvement by feature dimension
ax2 = axes[1]
gains = [(lda_accs[i] - full_accs[i]) * 100 for i in range(len(backbones))]

colors = [BACKBONE_COLORS[b] for b in backbones]
ax2.scatter(dims, gains, s=200, c=colors, edgecolor='black', linewidth=2)

for i, (dim, gain, backbone) in enumerate(zip(dims, gains, backbones)):
    ax2.annotate(backbone, (dim, gain), textcoords="offset points", 
                 xytext=(10, 5), fontsize=10)

ax2.axhline(y=0, color='gray', linestyle='--', linewidth=1)
ax2.set_xlabel('Feature Dimension')
ax2.set_ylabel('LDA Improvement (% points)')
ax2.set_title('LDA Gain vs Feature Dimensionality')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{FIGURE_DIR}/backbone_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nFigure saved: figures/backbone_comparison.png")

---
## 5. Accuracy vs Components Analysis

In [ ]:
# LDA vs PCA accuracy curves for one backbone
fig, ax = plt.subplots(figsize=(10, 6))

# Use ResNet-18 as example
backbone = 'ResNet-18'
backbone_data = backbone_results[backbone_results['backbone'] == backbone]

# Full features baseline
full_acc = backbone_data[backbone_data['method'] == 'None (Full)']['accuracy'].values[0]
ax.axhline(y=full_acc, color=METHOD_COLORS['Full'], linestyle='--', linewidth=2, label=f'Full Features ({full_acc:.4f})')

# LDA curve
lda_data = backbone_data[backbone_data['method'] == 'LDA'].sort_values('components')
ax.plot(lda_data['components'], lda_data['accuracy'], 'o-', 
        color=METHOD_COLORS['LDA'], linewidth=2, markersize=8, label='LDA')

# PCA curve
pca_data = backbone_data[backbone_data['method'] == 'PCA'].sort_values('components')
ax.plot(pca_data['components'], pca_data['accuracy'], 's-', 
        color=METHOD_COLORS['PCA'], linewidth=2, markersize=8, label='PCA')

ax.set_xlabel('Number of Components')
ax.set_ylabel('Test Accuracy')
ax.set_title(f'CIFAR-100 ({backbone}): Accuracy vs Components')
ax.legend(loc='lower right')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{FIGURE_DIR}/accuracy_vs_components.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nFigure saved: figures/accuracy_vs_components.png")

---
## 6. Summary Tables for Publication

In [ ]:
# Create publication-ready summary table
print("PUBLICATION SUMMARY TABLE")
print("="*80)

summary_data = []

for backbone in ['ResNet-18', 'MobileNetV3-Small', 'EfficientNet-B0', 'ResNet-50']:
    if backbone in full_baseline:
        dim = backbone_results[backbone_results['backbone'] == backbone]['feature_dim'].iloc[0]
        full_acc = full_baseline[backbone]
        lda_acc = best_lda[backbone]
        lda_comp = best_lda_comp[backbone]
        pca_acc = best_pca[backbone]
        gain_vs_full = (lda_acc - full_acc) * 100
        gain_vs_pca = (lda_acc - pca_acc) * 100
        
        summary_data.append({
            'Backbone': backbone,
            'Feature Dim': dim,
            'Full Features': f'{full_acc*100:.2f}%',
            'Best LDA': f'{lda_acc*100:.2f}% ({int(lda_comp)}c)',
            'Best PCA': f'{pca_acc*100:.2f}%',
            'LDA vs Full': f'{gain_vs_full:+.2f}%',
            'LDA vs PCA': f'{gain_vs_pca:+.2f}%'
        })

summary_df = pd.DataFrame(summary_data)
display(summary_df)

In [ ]:
# Statistical significance table
print("\nSTATISTICAL SIGNIFICANCE TABLE")
print("="*80)

sig_table = sig_summary[['dataset', 'n_components', 'acc_full_mean', 'acc_lda_mean', 
                          'lda_vs_full_mean', 'wilcoxon_pvalue_lda_vs_full']].copy()
sig_table.columns = ['Dataset', 'Components', 'Full Acc', 'LDA Acc', 'LDA Gain', 'p-value']
sig_table['LDA Gain'] = sig_table['LDA Gain'].apply(lambda x: f'{x*100:+.2f}%')
sig_table['Significant'] = sig_table['p-value'].apply(lambda x: '✓' if x < 0.05 else '')
sig_table['p-value'] = sig_table['p-value'].apply(lambda x: f'{x:.4f}')

display(sig_table)

---
## 7. Key Findings Summary

In [ ]:
print("\n" + "="*80)
print("KEY FINDINGS FROM LDA ANALYSIS")
print("="*80)

print("\n1. STATISTICAL SIGNIFICANCE:")
print("-"*40)
cifar_sig_row = sig_summary[sig_summary['dataset'] == 'cifar100'].iloc[0]
print(f"   • LDA improves CIFAR-100 accuracy by {cifar_sig_row['lda_vs_full_mean']*100:+.2f}%")
print(f"   • p-value = {cifar_sig_row['wilcoxon_pvalue_lda_vs_full']:.4f} (Wilcoxon signed-rank test)")
print(f"   • Result is STATISTICALLY SIGNIFICANT at p<0.05")

print("\n2. DATA EFFICIENCY:")
print("-"*40)
cifar_eff = eff_summary[eff_summary['dataset'] == 'cifar100'].sort_values('train_fraction')
gains = cifar_eff['lda_vs_full_mean'].values * 100
print(f"   • At 10% data: LDA gain = {gains[0]:+.2f}% (may underperform)")
print(f"   • At 50% data: LDA gain = {gains[2]:+.2f}% (strongest gain)")
print(f"   • At 100% data: LDA gain = {gains[3]:+.2f}% (consistent improvement)")

print("\n3. BACKBONE ANALYSIS:")
print("-"*40)
print("   • Smaller backbones (ResNet-18, MobileNetV3) show LARGER LDA gains")
print("   • Larger backbones (ResNet-50, EfficientNet) show smaller but still positive gains")
print("   • Hypothesis: Smaller backbones have more redundant features that LDA can exploit")

print("\n4. PRACTICAL IMPLICATIONS:")
print("-"*40)
print("   • LDA provides consistent accuracy improvements")
print("   • LDA enables significant dimensionality reduction (512D → 99D)")
print("   • Best suited for scenarios with sufficient training data (>25%)")
print("   • Especially beneficial for edge deployment on smaller backbones")

print("\n" + "="*80)